<center>
    <font size="5"> Sieci neuronowe i uczenie głębokie<br/>
        <small><em>Studia stacjonarne II stopnia 2025/2026</em><br/>Kierunek: Matematyka stosowana<br>Specjalność: Analityka danych</small>
    </font>
</center>
<br>



# Laboratorium nr 5: Detekcja obiektów

In [ ]:
!pip install keras-cv opencv-python

In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import xml.etree.ElementTree as ET
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import keras_cv

## Dataset: Fruit images, Jeden obiekt na obrazie, Model: VGG
Data: https://www.kaggle.com/datasets/mbkinaci/fruit-images-for-object-detection
### Wizualizacja danych

In [ ]:
IMAGE_DIR = './fruits/train/'       
ANNOTATION_DIR = './fruits/train/'

CLASS_MAPPING = {'apple': 0, 'banana': 1, 'orange': 2}

sample_xml = 'orange_15.xml' 
xml_path = os.path.join(ANNOTATION_DIR, sample_xml)

tree = ET.parse(xml_path)
root = tree.getroot()

filename = root.find('filename').text
image_path = os.path.join(IMAGE_DIR, filename)

obj = root.find('object')
class_name = obj.find('name').text

bndbox = obj.find('bndbox')
xmin = int(bndbox.find('xmin').text)
ymin = int(bndbox.find('ymin').text)
xmax = int(bndbox.find('xmax').text)
ymax = int(bndbox.find('ymax').text)

rect_w = xmax - xmin
rect_h = ymax - ymin

image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(image_rgb)

rect = patches.Rectangle(
    (xmin, ymin), rect_w, rect_h, 
    linewidth=3, edgecolor='red', facecolor='none'
)
ax.add_patch(rect)
ax.text(
    xmin, ymin - 10, class_name.capitalize(), 
    color='white', fontsize=14, fontweight='bold',
    bbox=dict(facecolor='red', alpha=0.8, edgecolor='none', pad=3)
)

plt.axis('off')
plt.title(f"Plik: {filename} | Klasa: {class_name}")
plt.tight_layout()
plt.show()

### Przygotowanie danych dla modelu

In [ ]:
TARGET_SIZE = (224, 224)

X_images = []  
y_classes = []
y_bboxes = [] 

xml_files = [f for f in os.listdir(ANNOTATION_DIR) if f.endswith('.xml')]

print(f"Znaleziono {len(xml_files)} plików XML.")

for xml_file in xml_files:
    xml_path = os.path.join(ANNOTATION_DIR, xml_file)
    
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    filename = root.find('filename').text
    image_path = os.path.join(IMAGE_DIR, filename)
    
    if not os.path.exists(image_path):
        continue
        
    image = cv2.imread(image_path)
    if image is None:
        continue
        
    # Zamiana kolorów z BGR (OpenCV domyślnie) na RGB (Keras domyślnie)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape
    
    # Szukamy pierwszego obiektu (zakładamy, że jest tylko jeden per zdjęcie)
    obj = root.find('object')
    if obj is None:
        continue
        
    class_name = obj.find('name').text.lower().strip()
    if class_name not in CLASS_MAPPING:
        continue
        
    bndbox = obj.find('bndbox')
    xmin = float(bndbox.find('xmin').text)
    ymin = float(bndbox.find('ymin').text)
    xmax = float(bndbox.find('xmax').text)
    ymax = float(bndbox.find('ymax').text)
    
    # Normalizacja współrzędnych do przedziału [0.0, 1.0]
    xmin_norm = xmin / w
    ymin_norm = ymin / h
    xmax_norm = xmax / w
    ymax_norm = ymax / h
    
    image_resized = cv2.resize(image, TARGET_SIZE)
    image_resized = image_resized.astype('float32') / 255.0
    #image_resized = tf.keras.applications.vgg16.preprocess_input(image_resized)
    
    X_images.append(image_resized)
    y_classes.append(CLASS_MAPPING[class_name])
    y_bboxes.append([xmin_norm, ymin_norm, xmax_norm, ymax_norm])


X_images = np.array(X_images)
y_bboxes = np.array(y_bboxes, dtype="float32")

# Dla klasyfikacji użyjemy One-Hot Encoding (np. [0, 1, 0] dla banana)
y_classes = to_categorical(y_classes, num_classes=len(CLASS_MAPPING))

print("\nPodsumowanie zbioru danych:")
print(f"Kształt X_images: {X_images.shape}")    # Powinno być (N, 224, 224, 3)
print(f"Kształt y_classes: {y_classes.shape}")  # Powinno być (N, 3)
print(f"Kształt y_bboxes: {y_bboxes.shape}")    # Powinno być (N, 4)

### Tworzenie modelu

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

vgg_backbone = VGG16(weights="imagenet", include_top=False, input_tensor=Input(shape=(224, 224, 3)))
vgg_backbone.trainable = False

flatten = Flatten()(vgg_backbone.output)
class_head = Dense(128, activation="relu")(flatten)
class_head = Dropout(0.5)(class_head) 
class_output = Dense(3, activation="softmax", name="class_output")(class_head)

box_head = Dense(128, activation="relu")(flatten)
box_head = Dense(64, activation="relu")(box_head)
box_output = Dense(4, activation="sigmoid", name="box_output")(box_head)

model = Model(inputs=vgg_backbone.input, outputs=[class_output, box_output])


model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss={
        "class_output": "categorical_crossentropy",
        "box_output": "mse"
    },
    metrics={
        "class_output": "accuracy",
        "box_output": "mse"
    }
    # Można też dodać loss_weights={'class_output': 1.0, 'box_output': 1.0}, jeśli jeden z członów dominuje
)

model.summary()

In [ ]:
INV_CLASS_MAPPING = {v: k for k, v in CLASS_MAPPING.items()}

def visualize_prediction(index, model):
    image = X_images[index]
    true_box = y_bboxes[index]
    
    true_class_idx = np.argmax(y_classes[index])
    true_class_name = INV_CLASS_MAPPING[true_class_idx]
    
    input_image = np.expand_dims(image, axis=0)
    pred_classes, pred_boxes = model.predict(input_image, verbose=0)
    
    pred_box = pred_boxes[0] 
    pred_class_idx = np.argmax(pred_classes[0])
    confidence = pred_classes[0][pred_class_idx]
    pred_class_name = INV_CLASS_MAPPING[pred_class_idx]
    
    h, w, _ = image.shape
    true_xmin, true_ymin, true_xmax, true_ymax = true_box * [w, h, w, h]
    pred_xmin, pred_ymin, pred_xmax, pred_ymax = pred_box * [w, h, w, h]
    
    fig, ax = plt.subplots(1, figsize=(7, 7))
    ax.imshow(image)
    
    true_rect = patches.Rectangle(
        (true_xmin, true_ymin), true_xmax - true_xmin, true_ymax - true_ymin,
        linewidth=2, edgecolor='lime', facecolor='none', linestyle='dashed', label='Prawda (XML)'
    )
    ax.add_patch(true_rect)
    
    pred_rect = patches.Rectangle(
        (pred_xmin, pred_ymin), pred_xmax - pred_xmin, pred_ymax - pred_ymin,
        linewidth=3, edgecolor='red', facecolor='none', label='Predykcja Modelu'
    )
    ax.add_patch(pred_rect)
    
    label_text = f"{pred_class_name.capitalize()} ({(confidence * 100):.1f}%)"
    ax.text(
        pred_xmin, pred_ymin - 5, label_text,
        color='white', fontsize=12, fontweight='bold',
        bbox=dict(facecolor='red', alpha=0.8, edgecolor='none', pad=2)
    )
    
    plt.legend(loc='upper right')
    plt.axis('off')
    plt.title(f"Wynik dla zdjęcia nr {index} | Prawdziwa klasa: {true_class_name.capitalize()}")
    plt.tight_layout()
    plt.show()


print("Generowanie wizualizacji...")
for _ in range(3):
    random_idx = np.random.randint(0, len(X_images) - 1)
    visualize_prediction(random_idx, model)

In [ ]:
history = model.fit(
    X_images,
    {
        "class_output": y_classes,
        "box_output": y_bboxes
    },
    validation_split=0.2, 
    batch_size=32,
    epochs=15             
)

# Zapisanie modelu po treningu
model.save("vgg16_fruit_detector.weights.h5")

In [ ]:
print("Generowanie wizualizacji...")
for _ in range(3):
    random_idx = np.random.randint(0, len(X_images) - 1)
    visualize_prediction(random_idx, model)

## Ćwiczenie 1
Przetestować model na danych testowych, ppoprawić w celu usunięcia efektu przeuczenia.

## Dataset: BCCD, Wiele obiektów na obrazie, Model: YOLO
Data: https://www.kaggle.com/datasets/orvile/bccd-blood-cell-count-and-detection-dataset/data
### Wizualizacja danych

In [ ]:
IMAGE_DIR = './bccd/train/img/'
ANNOTATION_DIR = './bccd/train/ann/'

sample_filename = 'BloodImage_00004.jpeg'

In [ ]:
image = Image.open(IMAGE_DIR+sample_filename)
print(image)
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(image)
    
with open(ANNOTATION_DIR+sample_filename+'.json' , 'r', encoding='utf-8') as f:
    data = json.load(f)
        
colors = {'RBC': 'red', 'WBC': 'blue', 'Platelets': 'green'}
    
for obj in data.get('objects', []):
    class_name = obj['classTitle']
        
    # Format bbox (xyxy): [[xmin, ymin], [xmax, ymax]]
    exterior_points = obj['points']['exterior']
    xmin, ymin = exterior_points[0]
    xmax, ymax = exterior_points[1]
        
    width = xmax - xmin
    height = ymax - ymin
        
    color = colors.get(class_name, 'yellow')
        
    rect = patches.Rectangle(
        (xmin, ymin), width, height, 
        linewidth=2, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(
        xmin, ymin - 5, class_name, 
        color=color, fontsize=12, fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
    )
    
plt.axis('off') # Wyłączenie osi współrzędnych
plt.title(f"Wizualizacja: {sample_filename}")
plt.tight_layout()
plt.show()

### Przygotowanie danych dla modelu YOLO z biblioteki keras_cv

In [ ]:
CLASS_MAPPING = {'RBC': 0, 'WBC': 1, 'Platelets': 2}

TARGET_SHAPE = (320, 320)
BATCH_SIZE = 4

In [ ]:
def parse_annotation(image_path, json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    boxes = []
    classes = []
    
    for obj in data.get('objects', []):
        class_name = obj['classTitle']
        
        if class_name not in CLASS_MAPPING:
            continue
            
        class_id = CLASS_MAPPING[class_name]
        
        # Oczekiwany format: [xmin, ymin, xmax, ymax] (tzw. format 'xyxy')
        exterior = obj['points']['exterior']
        xmin, ymin = exterior[0]
        xmax, ymax = exterior[1]
        
        boxes.append([xmin, ymin, xmax, ymax])
        classes.append(class_id)
        
    # Jeśli na obrazku nie ma obiektów, dodajemy pusty bounding box
    if len(boxes) == 0:
        boxes = tf.zeros((0, 4), dtype=tf.float32)
        classes = tf.zeros((0,), dtype=tf.float32)
    else:
        boxes = tf.cast(boxes, dtype=tf.float32)
        classes = tf.cast(classes, dtype=tf.float32)
        
    return image_path, boxes, classes

def data_generator():
    json_files = [f for f in os.listdir(ANNOTATION_DIR) if f.endswith('.json')]
    
    for json_file in json_files:
        image_filename = json_file.replace('.json', '')
        
        image_path = os.path.join(IMAGE_DIR, image_filename)
        json_path = os.path.join(ANNOTATION_DIR, json_file)
        
        if os.path.exists(image_path):
            img_path, boxes, classes = parse_annotation(image_path, json_path)
            yield img_path, boxes, classes

def load_image(image_path, boxes, classes):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    
    bounding_boxes = {
        "classes": classes,
        "boxes": boxes
    }
    return {"images": image, "bounding_boxes": bounding_boxes}

In [ ]:
dataset = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=(
        tf.TensorSpec(shape=(), dtype=tf.string),           # Ścieżka do pliku
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32),   # Bounding boxy (N, 4)
        tf.TensorSpec(shape=(None,), dtype=tf.float32)      # Klasy (N,)
    )
)

dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
resizing_layer = keras_cv.layers.Resizing(
    TARGET_SHAPE[0], TARGET_SHAPE[1], 
    bounding_box_format="xyxy", 
    pad_to_aspect_ratio=True 
)

# ragged_batch jest kluczowy, by móc ułożyć w jednym batchu obrazy o różnej liczbie ramek
dataset = dataset.map(resizing_layer, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.ragged_batch(BATCH_SIZE, drop_remainder=True)


# Przygotowanie do trenowania 
def dict_to_tuple(inputs):
    return inputs["images"], keras_cv.bounding_box.to_dense(inputs["bounding_boxes"], max_boxes=32)

dataset = dataset.map(dict_to_tuple, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = dataset.prefetch(tf.data.AUTOTUNE)

### Tworzenie modelu YOLO

In [ ]:
backbone = keras_cv.models.YOLOV8Backbone.from_preset("yolo_v8_xs_backbone_coco")  # Wersja "xs" 

model = keras_cv.models.YOLOV8Detector(
    num_classes=3,               # Nasze 3 klasy: RBC, WBC, Platelets
    bounding_box_format="xyxy",  # Ten sam format co w naszym datasecie
    backbone=backbone,
    fpn_depth=1                  # Głębokość piramidy cech
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    # Strata klasyfikacji: jak dobrze model odgaduje klasę w ramce
    classification_loss="binary_crossentropy",
    # Strata dla ramek: CIoU (Complete Intersection over Union) - ocenia jak dobrze ramka nakłada się na cel
    box_loss="ciou"
)


In [ ]:
CLASS_MAPPING_INV = {0: 'RBC', 1: 'WBC', 2: 'Platelets'}
images, ground_truth_boxes = next(iter(train_dataset.take(1)))
predictions = model.predict(images)
print("Generuję wizualizację...")
keras_cv.visualization.plot_bounding_box_gallery(
    images,
    value_range=(0, 255),               # Piksele w naszych obrazach są w zakresie 0-255
    bounding_box_format="xyxy",         # Format ramek, który konsekwentnie używamy
    y_true=ground_truth_boxes,          # Prawdziwe ramki (z JSON-ów)
    y_pred=predictions,                 # Ramki zgadnięte przez model YOLOv8
    scale=6,                            # Wielkość pojedynczego obrazka na wykresie
    rows=2,                             # Liczba wierszy w siatce (zależna od BATCH_SIZE, tu: 2x2=4)
    cols=2,                             # Liczba kolumn w siatce
    show=True,                          # Wyświetlenie obrazka (plt.show() pod spodem)
    font_scale=0.3,                     # Zmniejszona czcionka etykiet, by nie zasłaniała komórek
    class_mapping=CLASS_MAPPING_INV,    # Słownik zamieniający ID klas na ich nazwy
    line_thickness=1                    # Grubość ramek
)

In [ ]:
EPOCHS = 10 

history = model.fit(train_dataset,epochs=EPOCHS)

model.save_weights("yolov8_bccd_weights.weights.h5")

In [ ]:
images, ground_truth_boxes = next(iter(train_dataset.take(1)))
predictions = model.predict(images)
print("Generuję wizualizację...")
keras_cv.visualization.plot_bounding_box_gallery(
    images,
    value_range=(0, 255),
    bounding_box_format="xyxy",
    y_true=ground_truth_boxes,
    y_pred=predictions,
    scale=6,
    rows=2,
    cols=2,
    show=True,
    font_scale=0.3,
    class_mapping=CLASS_MAPPING_INV,
    line_thickness=1
)

## Ćwiczenie 2
Poprawić (dodać zbiór validacyjny i testowy) i dotrenować model aby osiągnąc lepsze wyniki (na zbiorze testowym).